# SEMAS — Rebuilt Reference Implementation

**Self-Adaptive Edge-Fog-Cloud Multi-Agent System** for Industrial IoT Predictive Maintenance.

This notebook accompanies the revised manuscript *"SEMAS: A Hierarchical Multi-Agent
Architecture with Online Policy Adaptation for Industrial IoT Predictive Maintenance"*
(IEEE Access). It replaces an earlier notebook whose PPO agent, federated aggregation,
LLM integration, and reported statistics did not match their described behavior.

**Design note.** This notebook imports the `semas/` package rather than redefining
agents inline. That package — not this notebook — is the single source of truth for
every number in the paper; it has been audited across five rounds for numeric
correctness (every table value traced to a `results/*.json` artifact), execution-order
correctness (no test-label leakage, verified at the code level), and reproducibility.

**Honest-evaluation invariants enforced throughout:**
1. Decision thresholds are calibrated on the **validation split only**, never on test labels.
2. Latency is measured end-to-end (feature vector in → decision out), boundary stated explicitly.
3. Every seed is wired through a single `set_seeds()` call — "N seeds" means N genuinely different runs.
4. All statistics are generated by code directly from per-seed logs. Nothing is hand-entered.
5. RUL is evaluated only on NASA C-MAPSS (genuine run-to-failure labels), never on synthetic proxies.

**Sections 1–5** run live (data, system build, PPO, baselines, SHAP+SLM demo — a few
minutes on a laptop CPU). **Sections 6–10** load the precomputed 5-seed sweep, ablations,
robustness suite, and CMAPSS/deep-baseline results (collectively several hours of CPU
time); a clearly marked cell in each section shows how to regenerate from scratch.


## 0. Setup

In [1]:
%pip install -q -r requirements.txt
# For Agent C (response generation): install Ollama from https://ollama.com,
# then run in a terminal:  ollama pull llama3.2:1b


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vinagent 0.0.5.post1 requires matplotlib==3.7.1, but you have matplotlib 3.11.1 which is incompatible.
vinagent 0.0.5.post1 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.


*(If pip prints "ERROR: ... dependency resolver ..." above, that is pip's standard, non-fatal conflict notice about an unrelated legacy package in this environment — it does not indicate installation failure.)*

In [2]:
import sys, os, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO_ROOT = Path("..").resolve() if not Path("semas").exists() else Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("Working directory:", Path.cwd())

import numpy as np
import pandas as pd

from semas.seeding import set_seeds
print("semas package importable — OK")


Working directory: <repository root>
semas package importable — OK


## 1. Datasets — honest task construction

Each loader documents and fixes a specific issue found in the original submission:

- **Boiler Emulator** (`semas/data/boiler.py`): the original label-encoding step marked
  only the "Fouling" class as anomalous, silently mislabeling Lean/ExcessAir/Scaling
  faults as normal. Corrected: anomaly = any Class ≠ Nominal, each with a graded
  severity parsed from the Condition field. Stratified 60/20/20 split (rows are
  independent i.i.d. operating points, not a time series).
- **Wind Turbine SCADA** (`semas/data/wind.py`): the original inner-joined SCADA to
  fault events on exact timestamps and undersampled to 500 rows (100 test samples),
  producing an F1>0.93 ceiling effect for every system. Corrected: the full
  37,120-record fault-log-covered series is retained, y(t)=1 if a fault occurs within
  a 60-minute horizon — a genuinely predictive label — with a **chronological**
  60/20/20 split (train precedes val precedes test in time).
- **NASA C-MAPSS FD001** (`semas/data/cmapss.py`): real run-to-failure RUL labels,
  replacing the original's synthetic, class-derived RUL targets (circular by
  construction).


In [3]:
from semas.data import load_boiler, load_wind, load_cmapss

set_seeds(42)

boiler = load_boiler(seed=42)
print(boiler.summary())

wind = load_wind()
print(wind.summary())
print("  train period:", wind.meta["train_period"])
print("  test period :", wind.meta["test_period"])

cmapss = load_cmapss()
for split in ["train", "val", "test"]:
    d = cmapss[split]
    print(f"cmapss {split}: n={len(d['RUL'])}, units={len(np.unique(d['unit']))}, "
          f"RUL mean={d['RUL'].mean():.1f}")


[boiler] train: n=1491, anomaly=29.2% | val: n=497, anomaly=29.2% | test: n=526, anomaly=33.1%
[wind] train: n=22272, anomaly=1.0% | val: n=7424, anomaly=5.9% | test: n=7424, anomaly=4.0%
  train period: ('2014-05-01 00:00:00', '2014-10-03 13:20:00')
  test period : ('2014-11-25 12:10:00', '2015-01-16 00:09:00')
cmapss train: n=16779, units=80, RUL mean=87.5
cmapss val: n=3852, units=20, RUL mean=84.1
cmapss test: n=13096, units=100, RUL mean=108.9


In [4]:
# Boiler severity-drift split: used later for the online-adaptation experiment.
# Anomalies are binned by ascending severity across 3 segments (class balance held
# constant ~28% per segment); normals are shuffled, not sorted, since severity=0 for
# every normal row by construction and a naive full-series sort would produce
# degenerate, all-normal early segments (a bug caught and fixed during this rebuild).
boiler_drift = load_boiler(seed=42, split="severity_drift")
print(boiler_drift.summary())


[boiler] train: n=972, anomaly=33.0% | val: n=324, anomaly=29.3% | test: n=1218, anomaly=27.8%


## 2. Building SEMAS (K=3 fog nodes)

- **Edge**: `EdgeFilter` — O(d) z-score pre-filter, cutoff tuned on validation data
  under an anomaly-pass-rate constraint (≥99% of true anomalies must still reach Fog).
- **Fog** (×K, disjoint training partitions): Agent B1 (Isolation Forest), Agent B2
  (5-model heterogeneous ensemble: IF, One-Class SVM, LOF, Elliptic Envelope, secondary
  IF; **soft/mean-score voting**, not majority voting), Agent B3 (weighted consensus).
- **Cloud**: Agent D (real PPO via stable-baselines3, not a heuristic), Agent E (real
  SHAP TreeExplainer attributions), and data-proportional **federated-style** parameter
  aggregation across the K nodes (never called "federated learning" — no privacy
  guarantee is made).


In [5]:
from semas.system import SemasSystem
from semas.evaluation import calibrate_threshold, classification_metrics, measure_latency
from semas.agents.fog_node import FogPolicy

set_seeds(42)
sem = SemasSystem(k_nodes=3, seed=42).fit(boiler.X_train.values)

z = sem.edge.tune(boiler.X_val.values, boiler.y_val)
print(f"edge filter tuned: z_cut={z:.2f}")

s_val = sem.scores(boiler.X_val.values)
tau0 = calibrate_threshold(boiler.y_val, s_val)
p = sem.global_policy()
sem.set_global_policy(FogPolicy(p.w1, p.contamination, tau0))
print(f"initial policy: w1={p.w1:.3f}, rho={p.contamination:.3f}, tau={tau0:.3f}")
print(f"edge filter rate on val: {sem.last_edge_filter_rate:.1%}")

m0 = classification_metrics(boiler.y_test, sem.scores(boiler.X_test.values), tau0)
print(f"test (pre-PPO): F1={m0['f1']:.3f} P={m0['precision']:.3f} "
      f"R={m0['recall']:.3f} AUC={m0['roc_auc']:.3f}")


edge filter tuned: z_cut=0.50
initial policy: w1=0.500, rho=0.300, tau=0.202
edge filter rate on val: 0.8%
test (pre-PPO): F1=0.531 P=0.404 R=0.776 AUC=0.647


## 3. PPO policy evolution (Agent D)

Real PPO (stable-baselines3: clipped surrogate objective, GAE), trained only on the
**validation** stream — the test set is never touched by adaptation. State
`[F1, Precision, Recall, FPR, w1, ρ, τ]`, bounded delta-actions, reward
`r = 0.4·F1 + 0.3·Precision − 0.2·FPR − 0.1·latency_norm` (the manuscript's Eq. 6,
identical to appendix Table XII — this was a reconciled inconsistency in the original).


In [6]:
from semas.agents.evolution import evolve_policy

model, best_policy = evolve_policy(sem, boiler.X_val.values, boiler.y_val,
                                    total_timesteps=512, seed=42, window=256)
print(f"evolved policy: w1={best_policy.w1:.3f}, rho={best_policy.contamination:.3f}, "
      f"tau={best_policy.tau:.3f}")

m1 = classification_metrics(boiler.y_test, sem.scores(boiler.X_test.values), best_policy.tau)
print(f"test (post-PPO): F1={m1['f1']:.3f} P={m1['precision']:.3f} "
      f"R={m1['recall']:.3f} AUC={m1['roc_auc']:.3f}")

lat = measure_latency(lambda X: sem.predict(X, tau=best_policy.tau),
                       boiler.X_test.values,
                       boundary="edge filter + routed fog node ensemble, CPU")
print(f"latency: {lat.per_sample_ms:.3f} ms/sample "
      f"(100 ms real-time budget — {'OK' if lat.per_sample_ms < 100 else 'VIOLATED'})")


evolved policy: w1=0.500, rho=0.300, tau=0.202
test (post-PPO): F1=0.531 P=0.404 R=0.776 AUC=0.647
latency: 0.443 ms/sample (100 ms real-time budget — OK)


## 4. Baselines (Baseline1 static, Baseline2 rule-based)

All three systems — Baseline1, Baseline2, and SEMAS — share the **identical detector
stack** (B1 Isolation Forest + B2 five-model ensemble + B3 weighted consensus), so any
measured difference isolates the adaptation/coordination mechanism, not detector choice.

Note documented in the manuscript: with a fixed `random_state`, Isolation Forest's
`score_samples` ranking is **provably invariant to the contamination parameter** —
contamination only shifts scikit-learn's internal classification offset, which this
system does not use externally-thresholded scores throughout instead. Baseline2's
effective adaptation levers are therefore its threshold rule and (newly added)
ensemble-weight-redistribution rule, not its contamination rule.


In [7]:
from semas.baselines.systems import Baseline1Static, Baseline2RuleBased

b1 = Baseline1Static(seed=42).fit(boiler.X_train.values, boiler.X_val.values, boiler.y_val)
b2 = Baseline2RuleBased(seed=42).fit(boiler.X_train.values, boiler.X_val.values, boiler.y_val)

for name, b in [("Baseline1", b1), ("Baseline2", b2)]:
    m = classification_metrics(boiler.y_test, b.scores(boiler.X_test.values), b.tau)
    print(f"{name}: F1={m['f1']:.3f} P={m['precision']:.3f} R={m['recall']:.3f} AUC={m['roc_auc']:.3f}")

print(f"SEMAS (PPO): F1={m1['f1']:.3f} P={m1['precision']:.3f} R={m1['recall']:.3f} AUC={m1['roc_auc']:.3f}")
print()
print("Single-seed demo only — see Section 6 for the full 5-seed statistical comparison.")


Baseline1: F1=0.540 P=0.399 R=0.833 AUC=0.645
Baseline2: F1=0.540 P=0.399 R=0.833 AUC=0.645
SEMAS (PPO): F1=0.531 P=0.404 R=0.776 AUC=0.647

Single-seed demo only — see Section 6 for the full 5-seed statistical comparison.


## 5. Explainability: Agent E (SHAP) grounding Agent C (SLM)

Agent E computes real SHAP (`TreeExplainer`) attributions for alerted samples. Agent C
generates a natural-language explanation with a **locally hosted** SLM (Llama-3.2-1B via
Ollama — no LLM, no cloud call), whose prompt is grounded in the top-3 SHAP features.

Requires a running Ollama server (`ollama serve`) with `llama3.2:1b` pulled. Skip this
cell if Ollama is not installed locally — the rest of the notebook does not depend on it.


In [8]:
from semas.agents.meta import AgentE

alerts = np.flatnonzero(sem.scores(boiler.X_test.values) >= best_policy.tau)
agent_e = AgentE(sem.nodes[0], boiler.feature_names, boiler.X_train.values)
scores_test = sem.scores(boiler.X_test.values)
top_alert = alerts[np.argmax(scores_test[alerts])]
attribution = agent_e.explain(boiler.X_test.values[[top_alert]])[0]

print(f"alert sample #{top_alert}, severity={scores_test[top_alert]:.2f}")
for feat, val in attribution:
    print(f"  {feat}: {val:+.3f}")


alert sample #293, severity=0.89
  temp_rise_per_fuel: -1.368
  temp_diff: -0.977
  Tsupply: -0.763


Background dataset has 200 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=200 when initializing the masker.


In [9]:
try:
    from semas.agents.response import AgentC, model_memory_mb
    agent_c = AgentC()
    rec = agent_c.generate(
        severity=float(scores_test[top_alert]),
        top_features=[f for f, _ in attribution],
        context="industrial boiler; supply/return temperature and fuel/water flow sensors",
    )
    print(f"latency: {rec['latency_s']:.1f}s | memory (ollama ps): {model_memory_mb()} MB")
    print()
    print(rec["response"])
except Exception as e:
    print(f"Ollama not reachable ({e}) — skipping live SLM call.")
    print("Manuscript-reported figures: Llama-3.2-1B, 1.52 GB resident, 35-82 s/response.")


latency: 11.0s | memory (ollama ps): 1519.292251 MB

Based on the provided data, a critical anomaly detected with a severity of 0.89 is likely related to the supply temperature and fuel/water flow sensors. The most probable failure mechanism involves a sudden increase in supply temperature due to excessive fuel or water flow, causing a thermal shock that may lead to equipment damage or complete failure. Immediate recommended action is to perform a diagnostic test on the affected sensor(s) while monitoring the system's overall performance. Required technician skills include expertise in boiler maintenance and troubleshooting. Expected intervention timeline: 2-4 hours.


## 6. Full comparative study (5 seeds × 3 systems × 3 conditions)

This is the source of every number in the manuscript's Performance Comparison and
Statistical Significance tables. It takes **on the order of hours** on a laptop CPU
(PPO evolution dominates the cost), so this notebook **loads the precomputed
`results/experiments.jsonl`** by default. To regenerate from scratch, run
`python scripts/run_experiments.py --seeds 5` from a terminal, or set
`REGENERATE = True` below (not recommended inside a notebook session).


In [10]:
REGENERATE = False

if REGENERATE:
    # Equivalent to: python scripts/run_experiments.py --seeds 5
    import subprocess
    subprocess.run([sys.executable, "scripts/run_experiments.py", "--seeds", "5"], check=True)

rows = [json.loads(l) for l in open("results/experiments.jsonl", encoding="utf-8")]
df = pd.DataFrame([{
    "dataset": r["dataset_key"], "system": r["system"], "seed": r["seed"],
    "f1": r["pooled"]["f1"], "precision": r["pooled"]["precision"],
    "recall": r["pooled"]["recall"], "roc_auc": r["pooled"]["roc_auc"],
    "delta_f1": r["delta_f1"], "latency_ms": r["per_sample_ms"],
} for r in rows])

summary = df.groupby(["dataset", "system"]).agg(
    n=("f1", "size"), f1_mean=("f1", "mean"), f1_std=("f1", "std"),
    precision_mean=("precision", "mean"), recall_mean=("recall", "mean"),
    auc_mean=("roc_auc", "mean"), auc_std=("roc_auc", "std"),
    delta_f1_mean=("delta_f1", "mean"), latency_ms=("latency_ms", "mean"),
).round(4)
summary


n  f1_mean  f1_std  ...  auc_std  delta_f1_mean  latency_ms
dataset       system                         ...                                    
boiler_drift  baseline1  5   0.5319  0.0117  ...   0.0095         0.0234      0.2779
              baseline2  5   0.5464  0.0148  ...   0.0095         0.0523      0.2666
              semas      5   0.5378  0.0377  ...   0.0123         0.0470      0.7798
boiler_static baseline1  5   0.5367  0.0185  ...   0.0169         0.0000      0.9373
              baseline2  5   0.5367  0.0185  ...   0.0169         0.0000      0.9076
              semas      5   0.5207  0.0595  ...   0.0251         0.0000      2.0334
wind_static   baseline1  5   0.0639  0.0017  ...   0.0093         0.0000      3.1684
              baseline2  5   0.0639  0.0017  ...   0.0093         0.0000      3.2717
              semas      5   0.0682  0.0172  ...   0.0040         0.0000      1.4999

[9 rows x 9 columns]

### Statistics — single source of truth (matches manuscript Table 9 exactly)

In [11]:
from scipy import stats as scipy_stats

print(f"{'Dataset':14s} {'Comparison':22s} {'dF1':>8s} {'t':>7s} {'p':>8s}  Significant?")
for dataset in sorted(df["dataset"].unique()):
    d = df[df["dataset"] == dataset]
    a = d[d["system"] == "semas"]["f1"].to_numpy()
    for base in ["baseline1", "baseline2"]:
        b = d[d["system"] == base]["f1"].to_numpy()
        if len(a) < 2 or len(b) < 2:
            continue
        t, p = scipy_stats.ttest_ind(a, b, equal_var=False)
        sig = "yes" if p < 0.05 else "no"
        print(f"{dataset:14s} SEMAS vs {base:10s} {a.mean()-b.mean():+8.4f} "
              f"{t:7.2f} {p:8.4f}  {sig}")

print()
print("Result: SEMAS's PPO-based adaptation is NOT statistically distinguishable from")
print("either baseline on detection F1 in any tested condition (all p > 0.5). This null")
print("result is reported directly in the manuscript rather than omitted or reframed.")


Dataset        Comparison                  dF1       t        p  Significant?
boiler_drift   SEMAS vs baseline1   +0.0059    0.33   0.7519  no
boiler_drift   SEMAS vs baseline2   -0.0086   -0.48   0.6536  no
boiler_static  SEMAS vs baseline1   -0.0160   -0.57   0.5926  no
boiler_static  SEMAS vs baseline2   -0.0160   -0.57   0.5926  no
wind_static    SEMAS vs baseline1   +0.0043    0.55   0.6088  no
wind_static    SEMAS vs baseline2   +0.0043    0.55   0.6088  no

Result: SEMAS's PPO-based adaptation is NOT statistically distinguishable from
either baseline on detection F1 in any tested condition (all p > 0.5). This null
result is reported directly in the manuscript rather than omitted or reframed.


## 7. Component ablation + volume-controlled K-ablation

Loaded from `results/ablation.json` (3 seeds, reduced PPO budget for tractability) and
`results/k_ablation.json` (5 seeds, no PPO — isolates node count K from per-node data
volume, since the naive K=1-vs-K=3 comparison confounds the two).


In [12]:
ablation = json.load(open("results/ablation.json", encoding="utf-8"))
for name, r in ablation.items():
    print(f"{name:20s} F1={r['f1_mean']:.4f} +- {r.get('f1_std', 0):.4f}")

print()
k_ablation = json.load(open("results/k_ablation.json", encoding="utf-8"))
for name, r in k_ablation.items():
    print(f"{name:20s} F1={r['f1_mean']:.4f} +- {r['f1_std']:.4f}  AUC={r['auc_mean']:.4f}")
print()
print("Volume-controlled result: no configuration is statistically distinguishable from")
print("any other (all pairwise Welch's t-tests p >= 0.72). The case for K>1 rests on")
print("architectural properties (locality, fault isolation, scaling), not accuracy.")


full                 F1=0.5323 +- 0.0104
no_ppo               F1=0.5323 +- 0.0104
no_consensus         F1=0.5172 +- 0.0068
no_federated         F1=0.4817 +- 0.0803

K3_partitioned       F1=0.5411 +- 0.0175  AUC=0.6536
K1_full              F1=0.5367 +- 0.0165  AUC=0.6575
K1_third             F1=0.5371 +- 0.0205  AUC=0.6539

Volume-controlled result: no configuration is statistically distinguishable from
any other (all pairwise Welch's t-tests p >= 0.72). The case for K>1 rests on
architectural properties (locality, fault isolation, scaling), not accuracy.


## 8. Robustness (sensor noise, missing data, reduced prevalence)

In [13]:
robustness = pd.DataFrame(json.load(open("results/robustness.json", encoding="utf-8")))
robustness.groupby(["condition", "system"])[["f1", "roc_auc"]].mean().round(3)


f1  roc_auc
condition       system                   
clean           baseline1  0.537    0.657
                semas      0.541    0.654
missing_0.1     baseline1  0.529    0.637
                semas      0.529    0.639
missing_0.3     baseline1  0.511    0.604
                semas      0.517    0.609
noise_0.1       baseline1  0.530    0.653
                semas      0.529    0.649
noise_0.3       baseline1  0.513    0.624
                semas      0.516    0.625
prevalence_0.05 baseline1  0.137    0.677
                semas      0.138    0.679
prevalence_0.15 baseline1  0.336    0.672
                semas      0.331    0.666

## 9. Deep-learning baselines and NASA C-MAPSS RUL

Loaded from `results/phase3.json` (supervised MLP/LSTM/Transformer classifiers +
CMAPSS RUL) and `results/ae_and_sensitivity.json` (corrected unsupervised autoencoder
baselines, re-run on the fixed data pipeline, plus the hyperparameter sensitivity sweep).


In [14]:
phase3 = json.load(open("results/phase3.json", encoding="utf-8"))
ae = json.load(open("results/ae_and_sensitivity.json", encoding="utf-8"))

print("Boiler:")
print(f"  Dense Autoencoder (unsup.): F1={ae['boiler_dense_ae']['f1']:.3f}  AUC={ae['boiler_dense_ae']['roc_auc']:.3f}")
print(f"  MLP classifier (sup.):      F1={phase3['boiler_mlp_supervised']['f1']:.3f}  AUC={phase3['boiler_mlp_supervised']['roc_auc']:.3f}")
print()
print("Wind:")
print(f"  LSTM Autoencoder (unsup.):  F1={ae['wind_lstm_ae']['f1']:.3f}  AUC={ae['wind_lstm_ae']['roc_auc']:.3f}")
print(f"  Transformer (sup.):         F1={phase3['wind_transformer_supervised']['f1']:.3f}  AUC={phase3['wind_transformer_supervised']['roc_auc']:.3f}")
print()
print("NASA C-MAPSS FD001 (real run-to-failure RUL labels):")
mae, rmse = phase3["cmapss_lstm_rul_lastcycle"]["mae"], phase3["cmapss_lstm_rul_lastcycle"]["rmse"]
print(f"  LSTM regressor: MAE={mae:.2f}, RMSE={rmse:.2f} cycles (last-cycle protocol)")
print(f"  Comparable published result: Zheng et al. 2017, RMSE=16.14 (same protocol)")


Boiler:
  Dense Autoencoder (unsup.): F1=0.547  AUC=0.689
  MLP classifier (sup.):      F1=0.906  AUC=0.941

Wind:
  LSTM Autoencoder (unsup.):  F1=0.172  AUC=0.557
  Transformer (sup.):         F1=0.063  AUC=0.569

NASA C-MAPSS FD001 (real run-to-failure RUL labels):
  LSTM regressor: MAE=11.20, RMSE=15.95 cycles (last-cycle protocol)
  Comparable published result: Zheng et al. 2017, RMSE=16.14 (same protocol)


## 10. Measured per-tier resources (not estimated)

Every figure below is measured via `psutil` process-RSS deltas or Ollama's own API on
this study's stated hardware — replacing the original submission's "(est.)" table.


In [15]:
res = json.load(open("results/resources.json", encoding="utf-8"))
for tier, vals in res.items():
    if tier == "host":
        continue
    print(f"{tier:12s} {vals}")
print()
print("host:", res["host"])


edge         {'rss_delta_mb': 0.0, 'model_size_mb': 0.000392, 'per_sample_ms': 0.0006056654020460471}
fog_node     {'rss_delta_mb': 11.95212799999996, 'model_size_mb': 10.102416, 'per_sample_ms': 1.2413804182598038}
cloud_ppo    {'rss_delta_mb': 79.41324800000001, 'policy_size_mb': 0.137762, 'update_64steps_s': 96.56791489999159}
cloud_shap   {'rss_delta_mb': 4.788224000000014, 'per_alert_ms': 120.09519999992335}
slm          {'resident_mb_ollama': 1519.292251, 'response_latency_s': 82.24201119999634, 'model': 'llama3.2:1b'}

host: {'cpu_count': 8, 'total_ram_gb': 17.006317568, 'process_rss_total_mb': 541.40928}


---
## Regenerating everything from scratch

```bash
python scripts/run_experiments.py --seeds 5     # Section 6 (hours)
python scripts/run_ablation.py                  # Section 7
python scripts/run_k_ablation.py                # Section 7
python scripts/run_robustness.py                # Section 8
python scripts/run_phase3.py                    # Section 9
python scripts/run_ae_and_sensitivity.py        # Section 9
python scripts/measure_resources.py             # Section 10
python scripts/analyze_stats.py                 # regenerates results/stats_table.md
```

See `README.md` for the full reproduction guide and dataset setup instructions.
